# 🃏 Arte das cartas v2 — estilo Hearthstone/Yu-Gi-Oh no T4

**O que mudou em relação ao notebook anterior:** o Juggernaut XL é um modelo treinado pra
**fotorrealismo** — ele sempre vai puxar pra foto, não importa o prompt. Aqui trocamos por
modelos ilustrados e adicionamos duas armas pesadas:

| Alavanca | Impacto | Onde |
|---|---|---|
| 1. Modelo ilustrado no lugar do fotorrealista | 🔥🔥🔥 | célula 4 |
| 2. Negativo anti-foto agressivo | 🔥 | célula 5 |
| 3. **Imagem de referência (IP-Adapter)** | 🔥🔥🔥🔥 | célula 6 |
| 4. LoRA de estilo Hearthstone | 🔥🔥 | célula 10 |

⭐ A **alavanca 3 é a que resolve de verdade**: você pega uma carta que o Nano Banana fez e
que você amou, e o SDXL copia o estilo dela. É o mesmo truque que funcionou lá, só que local
e de graça.

Antes de rodar: `Ambiente de execução` → `Alterar o tipo` → **GPU T4**.

## 1. Conferir a GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Sem GPU! Ambiente de execução → Alterar tipo → GPU T4"
print("\nGPU ok:", torch.cuda.get_device_name(0))

## 2. Instalar

In [ ]:
!pip -q install --upgrade diffusers transformers accelerate safetensors peft
print("pronto ✅")

## 3. Onde salvar

In [ ]:
import os

SALVAR_NO_DRIVE = True

if SALVAR_NO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_SAIDA = '/content/drive/MyDrive/card-game-arte'
else:
    PASTA_SAIDA = '/content/card-game-arte'

os.makedirs(PASTA_SAIDA, exist_ok=True)
print("Salvando em:", PASTA_SAIDA)

## 4. Escolher o modelo ilustrado

Todos rodam no T4 e todos são **muito** menos realistas que o Juggernaut.
Troque só a linha `ESCOLHA` e rode a célula de novo pra testar outro.

| Preset | Cara que ele dá | Velocidade no T4 |
|---|---|---|
| `pintado` | Fantasia pintada, o mais perto do Hearthstone | ~25s |
| `ilustrado` | Ilustração colorida, meio termo entre pintura e desenho | ~25s |
| `anime` | Anime forte — o lado Yu-Gi-Oh da mistura | ~25s |
| `rapido` | Versão turbo do ilustrado, pra iterar prompt voando | **~6s** ⚡ |

💡 Sugestão: comece no `rapido` pra achar o prompt, depois refaça as boas no `pintado`.

In [ ]:
import torch, gc
from diffusers import StableDiffusionXLPipeline, AutoencoderKL, DPMSolverMultistepScheduler

PRESETS = {
    "pintado":   dict(id="misri/zavychromaxl_v80",
                      extra="", steps=30, cfg=6.0),
    "ilustrado": dict(id="Lykon/dreamshaper-xl-1-0",
                      extra="", steps=30, cfg=6.0),
    "anime":     dict(id="cagliostrolab/animagine-xl-4.0",
                      extra="masterpiece, high score, great score, absurdres, ", steps=28, cfg=5.0),
    "rapido":    dict(id="Lykon/dreamshaper-xl-v2-turbo",
                      extra="", steps=8,  cfg=2.0),
}

ESCOLHA = "pintado"      # <<<<<< troque aqui: pintado | ilustrado | anime | rapido

P = PRESETS[ESCOLHA]
MODELO, EXTRA_QUALIDADE = P["id"], P["extra"]
STEPS, CFG = P["steps"], P["cfg"]

vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)

def carregar(nome):
    try:
        return StableDiffusionXLPipeline.from_pretrained(
            nome, vae=vae, torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
    except Exception:
        print("(sem variante fp16 nesse repo, baixando a padrão...)")
        return StableDiffusionXLPipeline.from_pretrained(
            nome, vae=vae, torch_dtype=torch.float16, use_safetensors=True)

try:
    del pipe; gc.collect(); torch.cuda.empty_cache()   # libera o modelo anterior
except NameError:
    pass

pipe = carregar(MODELO)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, algorithm_type="dpmsolver++", use_karras_sigmas=True)
pipe = pipe.to("cuda")

# ⚠️ NÃO usar pipe.enable_attention_slicing() aqui!
# Ele troca o processador de atenção por SlicedAttnProcessor, e o IP-Adapter da célula 6
# tenta recriar esse processador sem o argumento slice_size → TypeError.
# O PyTorch 2 já usa atenção eficiente (SDPA) por padrão, então não faz falta no T4.
pipe.enable_vae_slicing()

gc.collect(); torch.cuda.empty_cache()
print(f"\n✅ preset '{ESCOLHA}' → {MODELO}   |   steps={STEPS}  cfg={CFG}")

## 5. Estilo — agora com negativo anti-foto

O `NEGATIVO` é metade do trabalho aqui: ele empurra ativamente pra longe de foto,
cosplay e render 3D. **Não tire essas palavras.**

In [ ]:
ESTILO = ("2d digital illustration, hand painted, stylized fantasy trading card art, "
          "bold clean outlines, cel shaded, exaggerated heroic proportions, "
          "vivid saturated colors, dramatic rim lighting, painterly brush strokes, "
          "centered character, simple atmospheric background, dark vignette")

NEGATIVO = ("photo, photograph, photorealistic, realistic, hyperrealistic, dslr, 35mm, "
            "film grain, cosplay, real person, 3d render, octane render, unreal engine, cgi, "
            "text, watermark, signature, ui, border, frame, "
            "blurry, lowres, jpeg artifacts, bad anatomy, deformed hands, extra fingers, "
            "extra limbs, cropped head, out of frame")

LARGURA, ALTURA    = 832, 1216
IMAGENS_POR_PROMPT = 2

print("Estilo configurado ✅  (steps/cfg vêm do preset da célula 4)")

## 6. ⭐ Referência de estilo (IP-Adapter) — a alavanca mais forte

Aqui é onde a mágica acontece: você dá pro SDXL **uma imagem que o Nano Banana gerou** e
ele copia o estilo dela — paleta, pincelada, iluminação — aplicando no personagem novo.

**Como usar:** rode a célula, clique em `Escolher arquivos` e mande **uma** das cartas que
você mais gostou. O download do IP-Adapter (~2.5 GB) acontece só na primeira vez.

**Escala** (ajuste o `FORCA_REFERENCIA`):
- `0.3-0.4` → só um tempero de estilo
- `0.5-0.7` → ✅ ponto doce: mesmo estilo, personagem novo
- `0.8-1.0` → começa a copiar a composição e o próprio personagem da referência

Quer pular e testar sem referência? Deixe `USAR_REFERENCIA = False`.

In [ ]:
import io
from PIL import Image
from IPython.display import display

USAR_REFERENCIA  = True
FORCA_REFERENCIA = 0.6

REF = None
if USAR_REFERENCIA:
    from google.colab import files
    print("Mande UMA imagem de referência (aquela carta do Nano Banana que você amou):")
    enviados = files.upload()

    # lê os BYTES enviados (e não o nome do arquivo) — se você mandar a mesma imagem
    # duas vezes, o Colab salva como "nome (1).png" e abrir pelo nome pegaria a antiga
    nome_ref = list(enviados.keys())[0]
    REF = Image.open(io.BytesIO(enviados[nome_ref])).convert("RGB")

    # a referência não precisa ser gigante: o encoder reduz pra 224px de qualquer jeito
    REF.thumbnail((1024, 1024), Image.LANCZOS)

    # o IP-Adapter substitui os processadores de atenção, então qualquer slicing
    # ativo tem que sair antes (senão: SlicedAttnProcessor missing slice_size)
    try:
        pipe.disable_attention_slicing()
    except Exception:
        pass

    pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models",
                         weight_name="ip-adapter_sdxl.bin")
    pipe.set_ip_adapter_scale(FORCA_REFERENCIA)

    print(f"\n✅ referência '{nome_ref}' carregada (força {FORCA_REFERENCIA}):")
    display(REF.resize((REF.width // 3, REF.height // 3)))
else:
    try:
        pipe.unload_ip_adapter()
    except Exception:
        pass
    print("Sem referência — só prompt.")

## 7. Seus prompts

Cole aqui os 20 do arquivo `prompts_tank.py`. Comece com 2-3 pra calibrar o estilo.

In [ ]:
PROMPTS = {

    "tank_t1_vanguarda":      "young human footman in worn iron armor, plain round shield, "
                              "determined face, scarred cheek, muted steel and leather tones",

    "tank_t4_tita_de_bronze": "towering bronze titan warrior, verdigris patina armor, glowing molten "
                              "cracks, one hand raised in a challenging taunt, orange glow",

    # cole o resto aqui

}

print(f"{len(PROMPTS)} prompt(s) × {IMAGENS_POR_PROMPT} = {len(PROMPTS)*IMAGENS_POR_PROMPT} imagens")

## 8. Gerar 🎨

In [ ]:
import random, os, torch
from IPython.display import display

def gerar(prompts=None, seeds=None, quantas=None, mostrar=True):
    prompts = PROMPTS if prompts is None else prompts
    quantas = IMAGENS_POR_PROMPT if quantas is None else quantas
    feitas = []

    for nome, corpo in prompts.items():
        prompt_final = f"{EXTRA_QUALIDADE}{corpo}, {ESTILO}"
        for i in range(quantas):
            seed = seeds[i % len(seeds)] if seeds else random.randint(0, 2**31 - 1)
            g = torch.Generator("cuda").manual_seed(seed)

            kw = dict(prompt=prompt_final, negative_prompt=NEGATIVO,
                      width=LARGURA, height=ALTURA,
                      num_inference_steps=STEPS, guidance_scale=CFG, generator=g)
            if REF is not None:
                kw["ip_adapter_image"] = REF

            img = pipe(**kw).images[0]

            caminho = os.path.join(PASTA_SAIDA, f"{nome}__{ESCOLHA}__seed{seed}.png")
            img.save(caminho)
            feitas.append(caminho)

            print(f"✅ {nome}   seed = {seed}")
            if mostrar:
                display(img.resize((LARGURA // 2, ALTURA // 2)))
            torch.cuda.empty_cache()

    print(f"\n{len(feitas)} imagem(ns) em {PASTA_SAIDA}")
    return feitas


gerar()

## 9. Calibrar a força da referência

Gera a **mesma carta com a mesma seed** em 4 forças diferentes, pra você ver lado a lado
qual pega o estilo sem copiar o personagem. Rode uma vez e escolha o número.

In [ ]:
if REF is not None:
    nome, corpo = list(PROMPTS.items())[0]
    seed_fixa = 777

    for forca in [0.3, 0.5, 0.7, 0.9]:
        pipe.set_ip_adapter_scale(forca)
        img = pipe(prompt=f"{EXTRA_QUALIDADE}{corpo}, {ESTILO}", negative_prompt=NEGATIVO,
                   width=LARGURA, height=ALTURA, num_inference_steps=STEPS,
                   guidance_scale=CFG, ip_adapter_image=REF,
                   generator=torch.Generator("cuda").manual_seed(seed_fixa)).images[0]
        print(f"força = {forca}")
        display(img.resize((LARGURA // 3, ALTURA // 3)))
        torch.cuda.empty_cache()

    pipe.set_ip_adapter_scale(FORCA_REFERENCIA)   # volta pro valor da célula 6
else:
    print("Sem referência carregada — rode a célula 6 com USAR_REFERENCIA = True")

## 10. (Opcional) LoRA de estilo Hearthstone

Existe um LoRA público treinado em cartas de Hearthstone. Funciona melhor no preset
`pintado`/`ilustrado`, e a palavra-chave `Hearthstone Card` precisa estar no prompt.

⚠️ Ele às vezes desenha a **moldura da carta junto** — se acontecer, baixe o peso pra 0.5
ou reforce `no frame, no border` no negativo. Dá pra combinar com a referência da célula 6.

In [ ]:
USAR_LORA = True
PESO_LORA = 0.7

if USAR_LORA:
    pipe.load_lora_weights("Norod78/sdxl-hearthstone-card-style-lora",
                           weight_name="SDXL-HearthstoneCard-Lora.safetensors",
                           adapter_name="hs")
    pipe.set_adapters(["hs"], adapter_weights=[PESO_LORA])
    ESTILO = ESTILO + ", Hearthstone Card"
    print(f"✅ LoRA Hearthstone ativo (peso {PESO_LORA}) — rode a célula 8 de novo")
else:
    try:
        pipe.unload_lora_weights()
        ESTILO = ESTILO.replace(", Hearthstone Card", "")
        print("LoRA desligado")
    except Exception:
        pass

## 11. Refazer uma carta / baixar tudo

In [ ]:
# 4 variações de um prompt só
# gerar({"tank_t1_vanguarda": PROMPTS["tank_t1_vanguarda"]}, quantas=4)

# baixar tudo num zip (dispensável se salvou no Drive)
# import shutil; from google.colab import files
# shutil.make_archive('/content/cartas', 'zip', PASTA_SAIDA)
# files.download('/content/cartas.zip')

---
### 🩹 Ainda saiu realista? Ataque nesta ordem

1. **Confira o preset** — se `ESCOLHA` ficou em `pintado` e ainda está foto, vá pro `anime`,
   que é o mais distante de realismo que existe aqui.
2. **Suba a referência** (célula 6) pra `0.7`. É o que mais muda o resultado.
3. **Ligue o LoRA** (célula 10) junto com a referência.
4. **Corte palavras realistas do seu prompt**: `armor`, `warrior` e `soldier` são neutros,
   mas `detailed skin`, `photo`, `portrait`, `8k`, `sharp focus` puxam pra foto — tire.
5. **Baixe o CFG** pra 4.5-5. CFG alto endurece e "realiza" a imagem.

### Outros problemas

| Sintoma | Solução |
|---|---|
| `SlicedAttnProcessor missing slice_size` ao carregar o IP-Adapter | Alguma célula chamou `enable_attention_slicing()`. Rode `pipe.disable_attention_slicing()` e carregue de novo — a célula 6 já faz isso sozinha |
| `CUDA out of memory` | Use `IMAGENS_POR_PROMPT = 1`. Se insistir, rode `pipe.enable_model_cpu_offload()` — mais lento, mas cabe folgado (⚠️ **não** use `enable_attention_slicing`, quebra o IP-Adapter) |
| Imagem preta | Rode a célula 4 de novo (VAE fp16 não carregou) |
| Prompt ignorado no fim | Passou dos 77 tokens do SDXL — encurte o prompt ou o `ESTILO` |
| Copiou o personagem da referência | Força da referência alta demais → baixe pra 0.5 |
| Trocou de preset e deu erro estranho | Reinicie a sessão e rode do começo (modelo antigo preso na VRAM) |